In [ ]:
%pip install torchvision torch openml matplotlib thop hcrystalball

# notebook/trident/FLAML Demo - Overview.ipynb

In [ ]:
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

In [ ]:
df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(
        "wasbs://publicwasb@mmlspark.blob.core.windows.net/company_bankruptcy_prediction_data.csv"
    )
)
# print dataset size
print("records read: " + str(df.count()))

In [ ]:
display(df)

In [ ]:
train_raw, test_raw = df.randomSplit([0.8, 0.2], seed=41)

In [ ]:
from pyspark.ml.feature import VectorAssembler

feature_cols = df.columns[1:]
featurizer = VectorAssembler(inputCols=feature_cols, outputCol="features")
train_data = featurizer.transform(train_raw)["Bankrupt?", "features"]
test_data = featurizer.transform(test_raw)["Bankrupt?", "features"]

In [ ]:
import mlflow
mlflow.set_experiment("flaml_tune_demo")
mlflow.autolog(exclusive=False)

In [ ]:
def predict(model, test_data=test_data):

    predictions = model.transform(test_data)
    
    metrics = ComputeModelStatistics(
        evaluationMetric="classification",
        labelCol="Bankrupt?",
        scoredLabelsCol="prediction",
    ).transform(predictions)
    return metrics


In [ ]:
from synapse.ml.lightgbm import LightGBMClassifier
from synapse.ml.train import ComputeModelStatistics

with mlflow.start_run(run_name="tune_default") as run:
    model = LightGBMClassifier(objective="binary", featuresCol="features", labelCol="Bankrupt?", isUnbalance=True, dataTransferMode="bulk")
    model = model.fit(train_data)

    # Generate predictions and log metrics
    default_metrics = predict(model)
    
    mlflow.log_metrics({"accuracy": default_metrics.first()['accuracy'], "AUC": default_metrics.first()['AUC']})

    default_metrics.show()

In [ ]:
def train(lambdaL1, learningRate, numLeaves, numIterations, train_data=train_data, val_data=test_data):
    """
    This train() function:
     - takes hyperparameters as inputs (for tuning later)
     - returns the AUC score on the validation dataset

    Wrapping code as a function makes it easier to reuse the code later for tuning.
    """
    lgc = LightGBMClassifier(
        objective="binary",
        lambdaL1=lambdaL1,
        learningRate=learningRate,
        numLeaves=numLeaves,
        labelCol="Bankrupt?",
        numIterations=numIterations,
        isUnbalance=True,
        featuresCol="features",
    )
    model = lgc.fit(train_data)
    # Define an evaluation metric and evaluate the model on the validation dataset.
    eval_metric = predict(model, val_data)
    eval_metric = eval_metric.toPandas()['AUC'][0]
    return model, eval_metric

In [ ]:
import flaml
import time

# define the search space
params = {
    "lambdaL1": flaml.tune.uniform(0.001, 1),
    "learningRate": flaml.tune.uniform(0.001, 1),
    "numLeaves": flaml.tune.randint(30, 100),
    "numIterations": flaml.tune.randint(100, 300),
}

# define the tune function
def flaml_tune(config):
    _, metric = train(**config)
    return {"auc": metric}

In [ ]:
with mlflow.start_run(nested=True, run_name="Child Run: "):
    analysis = flaml.tune.run(
        flaml_tune,
        params,
        time_budget_s=60,
        num_samples=100,
        metric="auc",
        mode="max",
        verbose=2,
        force_cancel=True,
        )

In [ ]:
tune_config = analysis.best_config
tune_metrics_val = analysis.best_result
print("Best config: ", tune_config)
print("Best metrics on validation data: ", tune_metrics_val)

In [ ]:
tune_model, tune_metrics = train(train_data=train_data, val_data=test_data, **tune_config)
tune_metrics = predict(tune_model)
tune_metrics.show()

In [ ]:
''' import AutoML class from the FLAML package '''
from flaml import AutoML
from flaml.automl.spark.utils import to_pandas_on_spark
automl = AutoML()

In [ ]:
import mlflow
mlflow.autolog()
mlflow.set_experiment("automl_classification_demo")

In [ ]:
import os
settings = {
    "time_budget": 120,  # total running time in seconds
    "metric": 'roc_auc',
    "task": 'classification',  # task type
    "log_file_name": 'flaml_experiment.log',  # flaml log file
    "seed": 42,    # random seed
    "force_cancel": True,  # force stop training once time_budget is used up
    "mlflow_exp_name": "automl_classification_demo"
}

In [ ]:
df = to_pandas_on_spark(train_data)
type(df)

In [ ]:
'''The main flaml automl API'''

with mlflow.start_run(nested=True):
    automl.fit(dataframe=df, label='Bankrupt?', isUnbalance=True, **settings)

In [ ]:
''' retrieve best config'''
print('Best hyperparmeter config:', automl.best_config)
print('Best roc_auc on validation data: {0:.4g}'.format(1-automl.best_loss))
print('Training duration of best run: {0:.4g} s'.format(automl.best_config_train_time))

In [ ]:
automl_metrics = predict(automl.model.estimator)
automl_metrics.show()

In [ ]:
import mlflow
mlflow.autolog()
mlflow.set_experiment("automl_spark_demo")

In [ ]:
settings = {
    "time_budget": 120,  # total running time in seconds
    "metric": 'roc_auc',  # primary metrics for regression can be chosen from: ['mae','mse','r2','rmse','mape']
    "task": 'classification',  # task type    
    "seed": 7654321,    # random seed
    "use_spark": True,
    "n_concurrent_trials": 3,
    "force_cancel": True,
    "mlflow_exp_name": "automl_spark_demo"

}

In [ ]:
pandas_df = train_raw.toPandas()
pandas_df.head()

In [ ]:
'''The main flaml automl API'''
with mlflow.start_run(nested=True):
    automl.fit(dataframe=pandas_df, label='Bankrupt?', **settings)

In [ ]:
''' retrieve best config'''
print('Best hyperparmeter config:', automl.best_config)
print('Best roc_auc on validation data: {0:.4g}'.format(1-automl.best_loss))
print('Training duration of best run: {0:.4g} s'.format(automl.best_config_train_time))

# notebook/trident/automl_autolog_on.ipynb

In [ ]:
import time
import mlflow
import flaml
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

In [ ]:
automl_experiment = flaml.AutoML()
automl_settings = {
    "max_iter": 3,
    "metric": "r2",
    "task": "regression",
    "n_concurrent_trials": 2,
    "use_spark": True,  # use spark to parallelize the training
    "log_type": "better",  # flaml only logs better configs than the previous iters by default, set to "all" to log all trials
}
X, y = load_diabetes(return_X_y=True, as_frame=True)
train_x, test_x, train_y, test_y = train_test_split(X, y, test_size=0.25)
automl_experiment.fit(X_train=train_x, y_train=train_y, **automl_settings)

In [ ]:
print(automl_experiment.model)
print(automl_experiment.config_history)
print(automl_experiment.best_iteration)
print(automl_experiment.best_estimator)

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import RegressionEvaluator
from flaml.automl.spark.utils import to_pandas_on_spark

In [ ]:
pd_df = load_diabetes(as_frame=True).frame
df = spark.createDataFrame(pd_df)
df = df.repartition(4).cache()
df.count()
train, test = df.randomSplit([0.8, 0.2], seed=1)
feature_cols = df.columns[:-1]
featurizer = VectorAssembler(inputCols=feature_cols, outputCol="features")
train_data = featurizer.transform(train)["target", "features"]
test_data = featurizer.transform(test)["target", "features"]
automl = flaml.AutoML()
# no need to set use_spark since a spark model itself will run in parallel
settings = {
    "max_iter": 3,
    "metric": "mse",
    "task": "regression",  # task type
    "seed": 7654321,  # random seed
}
df = to_pandas_on_spark(train_data)

mlflow.set_experiment("automl_exp")
with mlflow.start_run(nested=True, run_name="automl_run"):
    automl.fit(dataframe=df, label="target", **settings)

model = automl.model.estimator
predictions = model.transform(test_data)
predictions.show(10)

evaluator = RegressionEvaluator(
    labelCol="target", predictionCol="prediction", metricName="mse"
)
metric = evaluator.evaluate(predictions)
print(f"mse: {metric}")

# notebook/trident/automl_autolog_off.ipynb

In [ ]:
import time
import mlflow
import flaml
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

In [ ]:
mlflow.autolog(disable=True)  # disable mlflow autologging

automl_experiment = flaml.AutoML()
automl_settings = {
    "max_iter": 3,
    "metric": "r2",
    "task": "regression",
    "n_concurrent_trials": 2,
    "use_spark": True,  # use spark to parallelize the training
    "log_type": "better",  # flaml only logs better configs than the previous iters by default, set to "all" to log all trials
}
X, y = load_diabetes(return_X_y=True, as_frame=True)
train_x, test_x, train_y, test_y = train_test_split(X, y, test_size=0.25)

with mlflow.start_run(nested=True):  # start a run to trigger pre-defined logging in FLAML
    automl_experiment.fit(X_train=train_x, y_train=train_y, **automl_settings)

In [ ]:
print(automl_experiment.model)
print(automl_experiment.config_history)
print(automl_experiment.best_iteration)
print(automl_experiment.best_estimator)

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import RegressionEvaluator
from flaml.automl.spark.utils import to_pandas_on_spark

In [ ]:
pd_df = load_diabetes(as_frame=True).frame
df = spark.createDataFrame(pd_df)
df = df.repartition(4).cache()
df.count()
train, test = df.randomSplit([0.8, 0.2], seed=1)
feature_cols = df.columns[:-1]
featurizer = VectorAssembler(inputCols=feature_cols, outputCol="features")
train_data = featurizer.transform(train)["target", "features"]
test_data = featurizer.transform(test)["target", "features"]
automl = flaml.AutoML()
# no need to set use_spark since a spark model itself will run in parallel
settings = {
    "max_iter": 3,
    "metric": "mse",
    "task": "regression",  # task type
    "seed": 7654321,  # random seed
}
df = to_pandas_on_spark(train_data)

mlflow.set_experiment("automl_exp")  # customize the experiment name
with mlflow.start_run(nested=True, run_name="automl_flaml"):  # customize the run name
    automl.fit(dataframe=df, label="target", labelCol="target", **settings)

model = automl.model.estimator
predictions = model.transform(test_data)
predictions.show(10)

evaluator = RegressionEvaluator(
    labelCol="target", predictionCol="prediction", metricName="mse"
)
metric = evaluator.evaluate(predictions)
print(f"mse: {metric}")

# notebook/trident/tune_autolog_on.ipynb

In [ ]:
import mlflow
import flaml
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing

import pyspark
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import RegressionEvaluator
from synapse.ml.lightgbm import LightGBMRegressor

In [ ]:
# Set pyspark autologging logModelAllowlist to include SynapseML models
spark.sparkContext._conf.set(
    "spark.mlflow.pysparkml.autolog.logModelAllowlistFile",
    "https://mmlspark.blob.core.windows.net/publicwasb/log_model_allowlist.txt",
)

In [ ]:
data = fetch_california_housing()

feature_cols = ["f" + str(i) for i in range(data.data.shape[1])]
header = ["target"] + feature_cols
df = spark.createDataFrame(pd.DataFrame(data=np.column_stack((data.target, data.data)), columns=header)).repartition(1)
print("Dataframe has {} rows".format(df.count()))

# Convert features into a single vector column
featurizer = VectorAssembler(inputCols=feature_cols, outputCol="features")
data = featurizer.transform(df)["target", "features"]

train_data, test_data = data.randomSplit([0.85, 0.15], seed=41)

In [ ]:
def train(config):
    """
    This train() function:
     - takes hyperparameters config as inputs (for tuning later)
     - returns the R^2 score on the test dataset

    Wrapping code as a function makes it easier to reuse the code later with FLAML.
    """
    lgr = LightGBMRegressor(
        objective="quantile",
        alpha=config["alpha"],
        learningRate=config["learningRate"],
        numLeaves=config["numLeaves"],
        labelCol="target",
        numIterations=config["numIterations"],
        dataTransferMode="bulk",
    )
    model = lgr.fit(train_data)
    # Define an evaluation metric and evaluate the model on the test dataset.
    predictions = model.transform(test_data)
    evaluator = RegressionEvaluator(predictionCol="prediction", labelCol="target", metricName="r2")
    eval_metric = evaluator.evaluate(predictions)

    return {"r2": eval_metric}

In [ ]:
params = {
    "alpha": flaml.tune.uniform(0, 1),
    "learningRate": flaml.tune.uniform(0.001, 1),
    "numLeaves": flaml.tune.randint(30, 100),
    "numIterations": flaml.tune.randint(100, 300),
}

In [ ]:
# no need to set use_spark since a spark model itself will run in parallel
analysis = flaml.tune.run(
    train,
    params,
    metric="r2",
    mode="max",
    num_samples=5,
)

In [ ]:
synapselgb_config = analysis.best_config
synapselgb_r2 = analysis.best_result['r2']
print(f"Best config: {synapselgb_config}")
print(f"R^2: {synapselgb_r2}")

In [ ]:
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor

X, y = fetch_california_housing(return_X_y=True, as_frame=True)
train_x, test_x, train_y, test_y = train_test_split(X, y, test_size=0.15)

In [ ]:
def train_lgb(config):
    lgr = LGBMRegressor(
        objective="quantile",
        alpha=config["alpha"],
        learningRate=config["learningRate"],
        numLeaves=config["numLeaves"],
        labelCol="target",
        numIterations=config["numIterations"],
    )
    model = lgr.fit(train_x, train_y, eval_metric=["l2"], eval_set=[(train_x, train_y)])
    # Define an evaluation metric and evaluate the model on the test dataset.
    pred_y = model.predict(test_x)
    r2 = r2_score(test_y, pred_y)

    return {"r2": r2}

In [ ]:
mlflow.set_experiment("tune_exp")  # customize the experiment name
with mlflow.start_run(nested=True, run_name="tune_run"):  # customize the run name
    analysis = flaml.tune.run(
        train_lgb,
        params,
        metric="r2",
        mode="max",
        num_samples=5,
        use_spark=True,  # use spark to parallelize the training
        n_concurrent_trials=2,
    )

In [ ]:
lgb_config = analysis.best_config
lgb_r2 = analysis.best_result['r2']
print(f"Best config: {lgb_config}")
print(f"R^2: {lgb_r2}")

# notebook/trident/tune_autolog_off.ipynb

In [ ]:
import mlflow
import flaml
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing

import pyspark
from pyspark.ml.feature import VectorAssembler
from synapse.ml.lightgbm import LightGBMRegressor
from synapse.ml.train import ComputeModelStatistics

mlflow.autolog(disable=True)  # disable mlflow autologging

In [ ]:
data = fetch_california_housing()

feature_cols = ["f" + str(i) for i in range(data.data.shape[1])]
header = ["target"] + feature_cols
df = spark.createDataFrame(pd.DataFrame(data=np.column_stack((data.target, data.data)), columns=header)).repartition(1)
print("Dataframe has {} rows".format(df.count()))

# Convert features into a single vector column
featurizer = VectorAssembler(inputCols=feature_cols, outputCol="features")
data = featurizer.transform(df)["target", "features"]

train_data, test_data = data.randomSplit([0.85, 0.15], seed=41)

In [ ]:
def train(config):
    """
    This train() function:
     - takes hyperparameters config as inputs (for tuning later)
     - returns the R^2 score on the test dataset

    Wrapping code as a function makes it easier to reuse the code later with FLAML.
    """
    lgr = LightGBMRegressor(
        objective="quantile",
        alpha=config["alpha"],
        learningRate=config["learningRate"],
        numLeaves=config["numLeaves"],
        labelCol="target",
        numIterations=config["numIterations"],
        dataTransferMode="bulk",
    )
    model = lgr.fit(train_data)
    # Define an evaluation metric and evaluate the model on the test dataset.
    predictions = model.transform(test_data)
    cms = ComputeModelStatistics(
        evaluationMetric="regression", labelCol="target", scoresCol="prediction"
    )
    metrics = cms.transform(predictions).collect()[0].asDict()

    # log metrics with mlflow
    with mlflow.start_run(nested=True):
        mlflow.log_metric("MSE", metrics["mean_squared_error"])
        mlflow.log_metric("RMSE", metrics["root_mean_squared_error"])
        mlflow.log_metric("R2", metrics["R^2"])
        mlflow.log_metric("MAE", metrics["mean_absolute_error"])

    return {"r2": metrics["R^2"]}

In [ ]:
params = {
    "alpha": flaml.tune.uniform(0, 1),
    "learningRate": flaml.tune.uniform(0.001, 1),
    "numLeaves": flaml.tune.randint(30, 100),
    "numIterations": flaml.tune.randint(100, 300),
}

In [ ]:
# no need to set use_spark since a spark model itself will run in parallel
analysis = flaml.tune.run(
    train,
    params,
    metric="r2",
    mode="max",
    num_samples=5,
)

mlflow.end_run()  # end current run

In [ ]:
synapselgb_config = analysis.best_config
synapselgb_r2 = analysis.best_result['r2']
print(f"Best config: {synapselgb_config}")
print(f"R^2: {synapselgb_r2}")

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor

X, y = fetch_california_housing(return_X_y=True, as_frame=True)
train_x, test_x, train_y, test_y = train_test_split(X, y, test_size=0.15)

In [ ]:
def train_lgb(config):
    lgr = LGBMRegressor(
        objective="quantile",
        alpha=config["alpha"],
        learningRate=config["learningRate"],
        numLeaves=config["numLeaves"],
        labelCol="target",
        numIterations=config["numIterations"],
    )
    model = lgr.fit(train_x, train_y, eval_metric=["l2"], eval_set=[(train_x, train_y)])
    # Define an evaluation metric and evaluate the model on the test dataset.
    pred_y = model.predict(test_x)
    r2 = r2_score(test_y, pred_y)
    mse = mean_squared_error(test_y, pred_y)
    mae = mean_absolute_error(test_y, pred_y)

    # # It's not recommended to log metrics manually and log flaml pre-defined metrics in the same time
    # # Below 4 lines of code is needed when the following manually logging is enabled and use_spark=True 
    # from synapse.ml.mlflow import set_mlflow_env_config
    # set_mlflow_env_config(mlflow_env_config)
    # if mlflow_exp_name is not None:
    #     mlflow.set_experiment(mlflow_exp_name)

    # with mlflow.start_run(nested=True):
    #     mlflow.log_metric("MSE", mse)
    #     mlflow.log_metric("R2", r2)
    #     mlflow.log_metric("MAE", mae)

    return {"r2": r2}

In [ ]:
mlflow_exp_name = "tune_exp"
mlflow.set_experiment(mlflow_exp_name)  # customize the experiment name
with mlflow.start_run(nested=True, run_name="tune_run"):  # customize the run name
    # # Below 2 lines of code is needed when the manually logging in train_lgb is enabled and use_spark=True 
    # from synapse.ml.mlflow import get_mlflow_env_config
    # mlflow_env_config = get_mlflow_env_config()

    analysis = flaml.tune.run(
        train_lgb,
        params,
        metric="r2",
        mode="max",
        num_samples=5,
        use_spark=True,  # use spark to parallelize the training
        n_concurrent_trials=2,
        mlflow_exp_name=mlflow_exp_name,
    )

In [ ]:
lgb_config = analysis.best_config
lgb_r2 = analysis.best_result['r2']
print(f"Best config: {lgb_config}")
print(f"R^2: {lgb_r2}")

# notebook/trident/demo_1_flight_delays_automl.ipynb

In [ ]:
from flaml.automl.data import load_openml_dataset
X_train, X_test, y_train, y_test = load_openml_dataset(dataset_id=1169, data_dir='./')

In [ ]:
X_train.head()

In [ ]:
''' import AutoML class from flaml package '''
from flaml import AutoML
automl = AutoML()

In [ ]:
settings = {
    "time_budget": 120,  # total running time in seconds
    "metric": 'accuracy', 
                        # check the documentation for options of metrics (https://microsoft.github.io/FLAML/docs/Use-Cases/Task-Oriented-AutoML#optimization-metric)
    "task": 'classification',  # task type
    "log_file_name": 'airlines_experiment.log',  # flaml log file
    "seed": 7654321,    # random seed
}


In [ ]:
'''The main flaml automl API'''
automl.fit(X_train=X_train, y_train=y_train, **settings)

In [ ]:
'''retrieve best config and best learner'''
print('Best ML leaner:', automl.best_estimator)
print('Best hyperparmeter config:', automl.best_config)
print('Best accuracy on validation data: {0:.4g}'.format(1-automl.best_loss))
print('Training duration of best run: {0:.4g} s'.format(automl.best_config_train_time))

In [ ]:
automl.model.estimator

In [ ]:
'''pickle and save the automl object'''
import pickle
with open('automl.pkl', 'wb') as f:
    pickle.dump(automl, f, pickle.HIGHEST_PROTOCOL)
'''load pickled automl object'''
with open('automl.pkl', 'rb') as f:
    automl = pickle.load(f)

In [ ]:
'''compute predictions of testing dataset''' 
y_pred = automl.predict(X_test)
print('Predicted labels', y_pred)
print('True labels', y_test)
y_pred_proba = automl.predict_proba(X_test)[:,1]

In [ ]:
''' compute different metric values on testing dataset'''
from flaml.ml import sklearn_metric_loss_score
print('accuracy', '=', 1 - sklearn_metric_loss_score('accuracy', y_pred, y_test))
print('roc_auc', '=', 1 - sklearn_metric_loss_score('roc_auc', y_pred_proba, y_test))
print('log_loss', '=', sklearn_metric_loss_score('log_loss', y_pred_proba, y_test))

In [ ]:
from flaml.automl.data import get_output_from_log
time_history, best_valid_loss_history, valid_loss_history, config_history, metric_history = \
    get_output_from_log(filename=settings['log_file_name'], time_budget=240)
for config in config_history:
    print(config)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.title('Learning Curve')
plt.xlabel('Wall Clock Time (s)')
plt.ylabel('Validation Accuracy')
plt.scatter(time_history, 1 - np.array(valid_loss_history))
plt.step(time_history, 1 - np.array(best_valid_loss_history), where='post')
plt.show()

In [ ]:
from lightgbm import LGBMClassifier
lgbm = LGBMClassifier()

In [ ]:
lgbm.fit(X_train, y_train)

In [ ]:
y_pred_lgbm = lgbm.predict(X_test)

In [ ]:
from xgboost import XGBClassifier
xgb = XGBClassifier()
cat_columns = X_train.select_dtypes(include=['category']).columns
X = X_train.copy()
X[cat_columns] = X[cat_columns].apply(lambda x: x.cat.codes)
y_train_xgb = y_train.astype("int")

In [ ]:
xgb.fit(X, y_train_xgb)

In [ ]:
X = X_test.copy()
X[cat_columns] = X[cat_columns].apply(lambda x: x.cat.codes)
y_pred_xgb = xgb.predict(X)
y_test_xgb = y_test.astype("int")


In [ ]:
print('default xgboost accuracy', '=', 1 - sklearn_metric_loss_score('accuracy', y_pred_xgb, y_test_xgb))
print('default lgbm accuracy', '=', 1 - sklearn_metric_loss_score('accuracy', y_pred_lgbm, y_test))
print('flaml (2 min) accuracy', '=', 1 - sklearn_metric_loss_score('accuracy', y_pred, y_test))

In [ ]:
!pip install rgf-python   

In [ ]:
''' SKLearnEstimator is the super class for a sklearn learner '''
from flaml.automl.model import SKLearnEstimator
from flaml import tune
from flaml.automl.task.task import CLASSIFICATION


class MyRegularizedGreedyForest(SKLearnEstimator):
    def __init__(self, task='binary', **config):
        '''Constructor
        
        Args:
            task: A string of the task type, one of
                'binary', 'multiclass', 'regression'
            config: A dictionary containing the hyperparameter names
                and 'n_jobs' as keys. n_jobs is the number of parallel threads.
        '''

        super().__init__(task, **config)

        '''task=binary or multi for classification task'''
        if task in CLASSIFICATION:
            from rgf.sklearn import RGFClassifier

            self.estimator_class = RGFClassifier
        else:
            from rgf.sklearn import RGFRegressor
            
            self.estimator_class = RGFRegressor

    @classmethod
    def search_space(cls, data_size, task):
        '''[required method] search space

        Returns:
            A dictionary of the search space. 
            Each key is the name of a hyperparameter, and value is a dict with
                its domain (required) and low_cost_init_value, init_value,
                cat_hp_cost (if applicable).
                e.g.,
                {'domain': tune.randint(lower=1, upper=10), 'init_value': 1}.
        '''
        space = {        
            'max_leaf': {'domain': tune.lograndint(lower=4, upper=data_size[0]), 'init_value': 4, 'low_cost_init_value': 4},
            'n_iter': {'domain': tune.lograndint(lower=1, upper=data_size[0]), 'init_value': 1, 'low_cost_init_value': 1},
            'n_tree_search': {'domain': tune.lograndint(lower=1, upper=32768), 'init_value': 1, 'low_cost_init_value': 1},
            'opt_interval': {'domain': tune.lograndint(lower=1, upper=10000), 'init_value': 100},
            'learning_rate': {'domain': tune.loguniform(lower=0.01, upper=20.0)},
            'min_samples_leaf': {'domain': tune.lograndint(lower=1, upper=20), 'init_value': 20},
        }
        return space

    @classmethod
    def size(cls, config):
        '''[optional method] memory size of the estimator in bytes
        
        Args:
            config - the dict of the hyperparameter config

        Returns:
            A float of the memory size required by the estimator to train the
            given config
        '''
        max_leaves = int(round(config['max_leaf']))
        n_estimators = int(round(config['n_iter']))
        return (max_leaves * 3 + (max_leaves - 1) * 4 + 1.0) * n_estimators * 8

    @classmethod
    def cost_relative2lgbm(cls):
        '''[optional method] relative cost compared to lightgbm
        '''
        return 1.0


In [ ]:
automl = AutoML()
automl.add_learner(learner_name='RGF', learner_class=MyRegularizedGreedyForest)

In [ ]:
settings = {
    "time_budget": 10,  # total running time in seconds
    "metric": 'accuracy', 
    "estimator_list": ['RGF', 'lgbm', 'rf', 'xgboost'],  # list of ML learners
    "task": 'classification',  # task type    
    "log_file_name": 'airlines_experiment_custom_learner.log',  # flaml log file 
    "log_training_metric": True,  # whether to log training metric
}

automl.fit(X_train=X_train, y_train=y_train, **settings)

In [ ]:
def custom_metric(X_val, y_val, estimator, labels, X_train, y_train,
                  weight_val=None, weight_train=None, config=None,
                  groups_val=None, groups_train=None):
    from sklearn.metrics import log_loss
    import time
    start = time.time()
    y_pred = estimator.predict_proba(X_val)
    pred_time = (time.time() - start) / len(X_val)
    val_loss = log_loss(y_val, y_pred, labels=labels,
                         sample_weight=weight_val)
    y_pred = estimator.predict_proba(X_train)
    train_loss = log_loss(y_train, y_pred, labels=labels,
                          sample_weight=weight_train)
    alpha = 0.5
    return val_loss * (1 + alpha) - alpha * train_loss, {
        "val_loss": val_loss, "train_loss": train_loss, "pred_time": pred_time
    }
    # two elements are returned:
    # the first element is the metric to minimize as a float number,
    # the second element is a dictionary of the metrics to log

In [ ]:
automl = AutoML()
settings = {
    "time_budget": 10,  # total running time in seconds
    "metric": custom_metric,  # pass the custom metric funtion here
    "task": 'classification',  # task type
    "log_file_name": 'airlines_experiment_custom_metric.log',  # flaml log file
}

automl.fit(X_train=X_train, y_train=y_train, **settings)

# notebook/trident/demo_2_house_price_tune_synapseml.ipynb

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing()

feature_cols = ["f" + str(i) for i in range(data.data.shape[1])]
header = ["target"] + feature_cols
df = spark.createDataFrame(
    pd.DataFrame(data=np.column_stack((data.target, data.data)), columns=header)
).repartition(1)

print("Dataframe has {} rows".format(df.count()))

In [ ]:
from pyspark.ml.feature import VectorAssembler

# Convert features into a single vector column
featurizer = VectorAssembler(inputCols=feature_cols, outputCol="features")
data = featurizer.transform(df)["target", "features"]

train_data, test_data = data.randomSplit([0.85, 0.15], seed=41)
train_data_sub, val_data_sub = train_data.randomSplit([0.85, 0.15], seed=41)

train_data.head()

In [ ]:
from synapse.ml.lightgbm import LightGBMRegressor
from pyspark.ml.evaluation import RegressionEvaluator

def train(alpha, learningRate, numLeaves, numIterations, train_data=train_data_sub, val_data=val_data_sub):
    """
    This train() function:
     - takes hyperparameters as inputs (for tuning later)
     - returns the R2 score on the validation dataset

    Wrapping code as a function makes it easier to reuse the code later for tuning.
    """

    lgr = LightGBMRegressor(
        objective="quantile",
        alpha=alpha,
        learningRate=learningRate,
        numLeaves=numLeaves,
        labelCol="target",
        numIterations=numIterations,
        dataTransferMode="bulk"
    )

    model = lgr.fit(train_data)

    # Define an evaluation metric and evaluate the model on the validation dataset.
    predictions = model.transform(val_data)
    evaluator = RegressionEvaluator(predictionCol="prediction", labelCol="target", metricName="r2")
    eval_metric = evaluator.evaluate(predictions)

    return model, eval_metric

In [ ]:
init_model, init_eval_metric = train(alpha=0.2, learningRate=0.3, numLeaves=31, numIterations=100, train_data=train_data, val_data=test_data)
print("R2 of initial model on test dataset is: ", init_eval_metric)

In [ ]:
import flaml
import time

# define the search space
params = {
    "alpha": flaml.tune.uniform(0, 1),
    "learningRate": flaml.tune.uniform(0.001, 1),
    "numLeaves": flaml.tune.randint(30, 100),
    "numIterations": flaml.tune.randint(100, 300),
}

# define the tune function
def flaml_tune(config):
    _, metric = train(**config)
    return {"r2": metric}

In [ ]:
analysis = flaml.tune.run(
    flaml_tune,
    params,
    time_budget_s=120,  # tuning in 120 seconds
    num_samples=100,
    metric="r2",
    mode="max",
    verbose=5,
    )

In [ ]:
flaml_config = analysis.best_config
print("Best config: ", flaml_config)

In [ ]:
flaml_model, flaml_metric = train(train_data=train_data, val_data=test_data, **flaml_config)

print("On the test dataset, the initial (untuned) model achieved R^2: ", init_eval_metric)
print("On the test dataset, the final flaml (tuned) model achieved R^2: ", flaml_metric)

# notebook/trident/demo_3_bankrupt_automl_synapseml.ipynb

In [ ]:
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

In [ ]:
df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(
        "wasbs://publicwasb@mmlspark.blob.core.windows.net/company_bankruptcy_prediction_data.csv"
    )
)
# print dataset size
print("records read: " + str(df.count()))

In [ ]:
display(df)

In [ ]:
train_raw, test_raw = df.randomSplit([0.8, 0.2], seed=41)

In [ ]:
from pyspark.ml.feature import VectorAssembler

feature_cols = df.columns[1:]
featurizer = VectorAssembler(inputCols=feature_cols, outputCol="features")
train_data = featurizer.transform(train_raw)["Bankrupt?", "features"]
test_data = featurizer.transform(test_raw)["Bankrupt?", "features"]

In [ ]:
from synapse.ml.lightgbm import LightGBMClassifier

model = LightGBMClassifier(
    objective="binary", featuresCol="features", labelCol="Bankrupt?", isUnbalance=True, dataTransferMode="bulk"
)

model = model.fit(train_data)

In [ ]:
def predict(model, test_data=test_data):
    from synapse.ml.train import ComputeModelStatistics

    predictions = model.transform(test_data)
    
    metrics = ComputeModelStatistics(
        evaluationMetric="classification",
        labelCol="Bankrupt?",
        scoredLabelsCol="prediction",
    ).transform(predictions)
    return metrics

default_metrics = predict(model)
default_metrics.show()

In [ ]:
train_data_sub, val_data_sub = train_data.randomSplit([0.8, 0.2], seed=41)

In [ ]:
def train(lambdaL1, learningRate, numLeaves, numIterations, train_data=train_data_sub, val_data=val_data_sub):
    """
    This train() function:
     - takes hyperparameters as inputs (for tuning later)
     - returns the AUC score on the validation dataset

    Wrapping code as a function makes it easier to reuse the code later for tuning.
    """

    lgc = LightGBMClassifier(
        objective="binary",
        lambdaL1=lambdaL1,
        learningRate=learningRate,
        numLeaves=numLeaves,
        labelCol="Bankrupt?",
        numIterations=numIterations,
        isUnbalance=True,
        featuresCol="features",
        dataTransferMode="bulk"
    )

    model = lgc.fit(train_data)

    # Define an evaluation metric and evaluate the model on the validation dataset.
    eval_metric = predict(model, val_data)
    eval_metric = eval_metric.toPandas()['AUC'][0]

    return model, eval_metric

In [ ]:
import flaml
import time

# define the search space
params = {
    "lambdaL1": flaml.tune.uniform(0.001, 1),
    "learningRate": flaml.tune.uniform(0.001, 1),
    "numLeaves": flaml.tune.randint(30, 100),
    "numIterations": flaml.tune.randint(100, 300),
}

# define the tune function
def flaml_tune(config):
    _, metric = train(**config)
    return {"auc": metric}

In [ ]:
analysis = flaml.tune.run(
    flaml_tune,
    params,
    time_budget_s=60,
    num_samples=100,
    metric="auc",
    mode="max",
    verbose=5,
    force_cancel=True,
    )

In [ ]:
tune_config = analysis.best_config
tune_metrics_val = analysis.best_result
print("Best config: ", tune_config)
print("Best metrics on validation data: ", tune_metrics_val)

In [ ]:
tune_model, tune_metrics = train(train_data=train_data, val_data=test_data, **tune_config)
tune_metrics = predict(tune_model)
tune_metrics.show()

In [ ]:
''' import AutoML class from the FLAML package '''
from flaml import AutoML
from flaml.automl.spark.utils import to_pandas_on_spark

automl = AutoML()

In [ ]:
import os
settings = {
    "time_budget": 60,  # total running time in seconds
    "metric": 'roc_auc',
    "task": 'classification',  # task type
    "log_file_name": 'flaml_experiment.log',  # flaml log file
    "seed": 42,    # random seed
    "force_cancel": True,  # force stop training once time_budget is used up
}

In [ ]:
df = to_pandas_on_spark(train_data)

type(df)

In [ ]:
'''The main flaml automl API'''
automl.fit(dataframe=df, label='Bankrupt?', isUnbalance=True, **settings)

In [ ]:
''' retrieve best config'''
print('Best hyperparmeter config:', automl.best_config)
print('Best roc_auc on validation data: {0:.4g}'.format(1-automl.best_loss))
print('Training duration of best run: {0:.4g} s'.format(automl.best_config_train_time))

In [ ]:
automl_metrics = predict(automl.model.estimator)
automl_metrics.show()

In [ ]:
settings = {
    "time_budget": 60,  # total running time in seconds
    "metric": 'roc_auc',  # primary metrics for regression can be chosen from: ['mae','mse','r2','rmse','mape']
    "task": 'classification',  # task type    
    "seed": 7654321,    # random seed
    "use_spark": True,
    "n_concurrent_trials": 2,
    "force_cancel": True,
}

In [ ]:
pandas_df = train_raw.toPandas()
pandas_df.head()

In [ ]:
'''The main flaml automl API'''
automl.fit(dataframe=pandas_df, label='Bankrupt?', **settings)

In [ ]:
''' retrieve best config'''
print('Best hyperparmeter config:', automl.best_config)
print('Best roc_auc on validation data: {0:.4g}'.format(1-automl.best_loss))
print('Training duration of best run: {0:.4g} s'.format(automl.best_config_train_time))

In [ ]:
# predict function for non-spark models
def predict_pandas(automl, test_raw):
    from synapse.ml.train import ComputeModelStatistics
    import pandas as pd
    pandas_test = test_raw.toPandas()
    predictions = automl.predict(pandas_test.iloc[:,1:]).astype('float')
    predictions = pd.DataFrame({"Bankrupt?":pandas_test.iloc[:,0], "prediction": predictions.tolist()})
    predictions = spark.createDataFrame(predictions)
    
    metrics = ComputeModelStatistics(
        evaluationMetric="classification",
        labelCol="Bankrupt?",
        scoredLabelsCol="prediction",
    ).transform(predictions)
    return metrics

automl_metrics = predict_pandas(automl, test_raw)
automl_metrics.show()

# notebook/trident/demo_4_tune_lexicographic.ipynb

In [ ]:
import torch
import thop
import torch.nn as nn
from flaml import tune
import torch.nn.functional as F
import torchvision
import numpy as np
import os

DEVICE = torch.device("cpu")
BATCHSIZE = 128
N_TRAIN_EXAMPLES = BATCHSIZE * 30
N_VALID_EXAMPLES = BATCHSIZE * 10
data_dir = os.path.abspath("data")

train_dataset = torchvision.datasets.FashionMNIST(
    data_dir,
    train=True,
    download=True,
    transform=torchvision.transforms.ToTensor(),
)

train_loader = torch.utils.data.DataLoader(
    torch.utils.data.Subset(train_dataset, list(range(N_TRAIN_EXAMPLES))),
    batch_size=BATCHSIZE,
    shuffle=True,
)

val_dataset = torchvision.datasets.FashionMNIST(
    data_dir, train=False, transform=torchvision.transforms.ToTensor()
)

val_loader = torch.utils.data.DataLoader(
    torch.utils.data.Subset(val_dataset, list(range(N_VALID_EXAMPLES))),
    batch_size=BATCHSIZE,
    shuffle=True,
)

In [ ]:
def define_model(configuration):
    n_layers = configuration["n_layers"]
    layers = []
    in_features = 28 * 28
    for i in range(n_layers):
        out_features = configuration["n_units_l{}".format(i)]
        layers.append(nn.Linear(in_features, out_features))
        layers.append(nn.ReLU())
        p = configuration["dropout_{}".format(i)]
        layers.append(nn.Dropout(p))
        in_features = out_features
    layers.append(nn.Linear(in_features, 10))
    layers.append(nn.LogSoftmax(dim=1))
    return nn.Sequential(*layers)

In [ ]:
def train_model(model, optimizer, train_loader):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.view(-1, 28 * 28).to(DEVICE), target.to(DEVICE)
        optimizer.zero_grad()
        F.nll_loss(model(data), target).backward()
        optimizer.step()

In [ ]:
def eval_model(model, valid_loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(valid_loader):
            data, target = data.view(-1, 28 * 28).to(DEVICE), target.to(DEVICE)
            pred = model(data).argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    accuracy = correct / N_VALID_EXAMPLES
    flops, params = thop.profile(
        model, inputs=(torch.randn(1, 28 * 28).to(DEVICE),), verbose=False
    )
    return np.log2(flops), 1 - accuracy, params

In [ ]:
def evaluate_function(configuration):
    model = define_model(configuration).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), configuration["lr"])
    n_epoch = configuration["n_epoch"]
    for epoch in range(n_epoch):
        train_model(model, optimizer, train_loader)
    flops, error_rate, params = eval_model(model, val_loader)
    return {"error_rate": error_rate, "flops": flops, "params": params}

In [ ]:
lexico_objectives = {}
lexico_objectives["metrics"] = ["error_rate", "flops"]
lexico_objectives["tolerances"] = {"error_rate": 0.02, "flops": 0.0}
lexico_objectives["targets"] = {"error_rate": 0.0, "flops": 0.0}
lexico_objectives["modes"] = ["min", "min"]

In [ ]:
search_space = {
    "n_layers": tune.randint(lower=1, upper=3),
    "n_units_l0": tune.randint(lower=4, upper=128),
    "n_units_l1": tune.randint(lower=4, upper=128),
    "n_units_l2": tune.randint(lower=4, upper=128),
    "dropout_0": tune.uniform(lower=0.2, upper=0.5),
    "dropout_1": tune.uniform(lower=0.2, upper=0.5),
    "dropout_2": tune.uniform(lower=0.2, upper=0.5),
    "lr": tune.loguniform(lower=1e-5, upper=1e-1),
    "n_epoch": tune.randint(lower=1, upper=20),
}

In [ ]:
low_cost_partial_config = {
    "n_layers": 1,
    "n_units_l0": 4,
    "n_units_l1": 4,
    "n_units_l2": 4,
    "n_epoch": 1,
}

analysis = tune.run(
    evaluate_function,
    num_samples=-1,
    time_budget_s=100,
    config=search_space,
    use_spark=True,
    lexico_objectives=lexico_objectives,
    low_cost_partial_config=low_cost_partial_config,
)
result = analysis.best_result
print(result)

# notebook/automl_time_series_forecast.ipynb

In [ ]:
import statsmodels.api as sm
data = sm.datasets.co2.load_pandas().data
# data is given in weeks, but the task is to predict monthly, so use monthly averages instead
data = data['co2'].resample('MS').mean()
data = data.bfill().ffill()  # makes sure there are no missing values
data = data.to_frame().reset_index()

In [ ]:
# split the data into a train dataframe and X_test and y_test dataframes, where the number of samples for test is equal to
# the number of periods the user wants to predict
num_samples = data.shape[0]
time_horizon = 12
split_idx = num_samples - time_horizon
train_df = data[:split_idx]  # train_df is a dataframe with two columns: timestamp and label
X_test = data[split_idx:]['index'].to_frame()  # X_test is a dataframe with dates for prediction
y_test = data[split_idx:]['co2']  # y_test is a series of the values corresponding to the dates for prediction

In [ ]:
train_df

import matplotlib.pyplot as plt

plt.plot(train_df['index'], train_df['co2'])
plt.xlabel('Date')
plt.ylabel('CO2 Levels')
plt.show()

In [ ]:
''' import AutoML class from flaml package '''
from flaml import AutoML
automl = AutoML()

In [ ]:
settings = {
    "time_budget": 240,  # total running time in seconds
    "metric": 'mape',  # primary metric for validation: 'mape' is generally used for forecast tasks
    "task": 'ts_forecast',  # task type
    "log_file_name": 'CO2_forecast.log',  # flaml log file
    "eval_method": "holdout",  # validation method can be chosen from ['auto', 'holdout', 'cv']
    "seed": 7654321,  # random seed
}

In [ ]:
'''The main flaml automl API'''
automl.fit(dataframe=train_df,  # training data
           label='co2',  # label column
           period=time_horizon,  # key word argument 'period' must be included for forecast task)
           **settings)

In [ ]:
''' retrieve best config and best learner'''
print('Best ML leaner:', automl.best_estimator)
print('Best hyperparmeter config:', automl.best_config)
print(f'Best mape on validation data: {automl.best_loss}')
print(f'Training duration of best run: {automl.best_config_train_time}s')

In [ ]:
automl.model.estimator

In [ ]:
''' pickle and save the automl object '''
import pickle
with open('automl.pkl', 'wb') as f:
    pickle.dump(automl, f, pickle.HIGHEST_PROTOCOL)

In [ ]:
''' compute predictions of testing dataset '''
flaml_y_pred = automl.predict(X_test)
print(f"Predicted labels\n{flaml_y_pred}")
print(f"True labels\n{y_test}")

In [ ]:
''' compute different metric values on testing dataset'''
from flaml.ml import sklearn_metric_loss_score
print('mape', '=', sklearn_metric_loss_score('mape', y_true=y_test, y_predict=flaml_y_pred))

In [ ]:
from flaml.automl.data import get_output_from_log
time_history, best_valid_loss_history, valid_loss_history, config_history, train_loss_history = \
    get_output_from_log(filename=settings['log_file_name'], time_budget=180)

for config in config_history:
    print(config)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.title('Learning Curve')
plt.xlabel('Wall Clock Time (s)')
plt.ylabel('Validation Accuracy')
plt.scatter(time_history, 1 - np.array(valid_loss_history))
plt.step(time_history, 1 - np.array(best_valid_loss_history), where='post')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
plt.plot(X_test, y_test, label='Actual level')
plt.plot(X_test, flaml_y_pred, label='FLAML forecast')
plt.xlabel('Date')
plt.ylabel('CO2 Levels')
plt.legend()

In [ ]:
''' multivariate time series forecasting dataset'''
import pandas as pd
# pd.set_option("display.max_rows", None, "display.max_columns", None)
multi_df = pd.read_csv(
    "https://raw.githubusercontent.com/srivatsan88/YouTubeLI/master/dataset/nyc_energy_consumption.csv"
)
# preprocessing data
multi_df["timeStamp"] = pd.to_datetime(multi_df["timeStamp"])
multi_df = multi_df.set_index("timeStamp")
multi_df = multi_df.resample("D").mean()
multi_df["temp"] = multi_df["temp"].fillna(method="ffill")
multi_df["precip"] = multi_df["precip"].fillna(method="ffill")
multi_df = multi_df[:-2]  # last two rows are NaN for 'demand' column so remove them
multi_df = multi_df.reset_index()

In [ ]:
''' Use feature engineering to create a categorical value'''
# Using temperature values create categorical values 
# where 1 denotes daily tempurature is above monthly average and 0 is below.

def get_monthly_avg(data):
    data["month"] = data["timeStamp"].dt.month
    data = data[["month", "temp"]].groupby("month")
    data = data.agg({"temp": "mean"})
    return data

monthly_avg = get_monthly_avg(multi_df).to_dict().get("temp")

def above_monthly_avg(date, temp):
    month = date.month
    if temp > monthly_avg.get(month):
        return 1
    else:
        return 0

multi_df["temp_above_monthly_avg"] = multi_df.apply(
    lambda x: above_monthly_avg(x["timeStamp"], x["temp"]), axis=1
)

del multi_df["month"]  # remove temperature column to reduce redundancy

In [ ]:
# split data into train and test
num_samples = multi_df.shape[0]
multi_time_horizon = 180
split_idx = num_samples - multi_time_horizon
multi_train_df = multi_df[:split_idx]
multi_test_df = multi_df[split_idx:]

multi_X_test = multi_test_df[
    ["timeStamp", "precip", "temp", "temp_above_monthly_avg"]
]  # test dataframe must contain values for the regressors / multivariate variables
multi_y_test = multi_test_df["demand"]

multi_train_df

In [ ]:
from flaml import AutoML
automl = AutoML()
settings = {
    "time_budget": 10,  # total running time in seconds
    "metric": "mape",  # primary metric
    "task": "ts_forecast",  # task type
    "log_file_name": "energy_forecast_categorical.log",  # flaml log file
    "eval_method": "holdout",
    "log_type": "all",
    "label": "demand",
}
'''The main flaml automl API'''
try:
    import prophet

    automl.fit(dataframe=multi_train_df, **settings, period=multi_time_horizon)
except ImportError:
    print("not using prophet due to ImportError")
    automl.fit(
        dataframe=multi_train_df,
        **settings,
        estimator_list=["arima", "sarimax"],
        period=multi_time_horizon,
    )

In [ ]:
''' compute predictions of testing dataset '''
multi_y_pred = automl.predict(multi_X_test)
print("Predicted labels", multi_y_pred)
print("True labels", multi_y_test)

In [ ]:
''' compute different metric values on testing dataset'''
from flaml.ml import sklearn_metric_loss_score
print('mape', '=', sklearn_metric_loss_score('mape', y_true=multi_y_test, y_predict=multi_y_pred))

In [ ]:
import matplotlib.pyplot as plt
plt.figure()
plt.plot(multi_X_test["timeStamp"], multi_y_test, label="Actual Demand")
plt.plot(multi_X_test["timeStamp"], multi_y_pred, label="FLAML Forecast")
plt.xlabel("Date")
plt.ylabel("Energy Demand")
plt.legend()
plt.show()

In [ ]:
from hcrystalball.utils import get_sales_data
time_horizon = 30
df = get_sales_data(n_dates=180, n_assortments=1, n_states=1, n_stores=1)
df = df[["Sales", "Open", "Promo", "Promo2"]]
# feature engineering - create a discrete value column
# 1 denotes above mean and 0 denotes below mean
import numpy as np
df["above_mean_sales"] = np.where(df["Sales"] > df["Sales"].mean(), 1, 0)
df.reset_index(inplace=True)
# train-test split
discrete_train_df = df[:-time_horizon]
discrete_test_df = df[-time_horizon:]
discrete_X_train, discrete_X_test = (
    discrete_train_df[["Date", "Open", "Promo", "Promo2"]],
    discrete_test_df[["Date", "Open", "Promo", "Promo2"]],
)
discrete_y_train, discrete_y_test = discrete_train_df["above_mean_sales"], discrete_test_df["above_mean_sales"]

In [ ]:
discrete_train_df

In [ ]:
from flaml import AutoML
automl = AutoML()

In [ ]:
settings = {
    "time_budget": 15,  # total running time in seconds
    "metric": "accuracy",  # primary metric
    "task": "ts_forecast_classification",  # task type
    "log_file_name": "sales_classification_forecast.log",  # flaml log file
    "eval_method": "holdout",
}

In [ ]:
"""The main flaml automl API"""
automl.fit(X_train=discrete_X_train,
           y_train=discrete_y_train,
           **settings,
           period=time_horizon)

In [ ]:
""" retrieve best config and best learner"""
print("Best ML leaner:", automl.best_estimator)
print("Best hyperparmeter config:", automl.best_config)
print(f"Best mape on validation data: {automl.best_loss}")
print(f"Training duration of best run: {automl.best_config_train_time}s")
print(automl.model.estimator)

In [ ]:
""" compute predictions of testing dataset """
discrete_y_pred = automl.predict(discrete_X_test)
print("Predicted label", discrete_y_pred)
print("True label", discrete_y_test)

In [ ]:
from flaml.ml import sklearn_metric_loss_score
print("accuracy", "=", 1 - sklearn_metric_loss_score("accuracy", discrete_y_test, discrete_y_pred))

In [ ]:
%pip install pytorch-forecasting==1.0.0

In [ ]:
def get_stalliion_data():
    from pytorch_forecasting.data.examples import get_stallion_data

    data = get_stallion_data()
    # add time index
    data["time_idx"] = data["date"].dt.year * 12 + data["date"].dt.month
    data["time_idx"] -= data["time_idx"].min()
    # add additional features
    data["month"] = data.date.dt.month.astype(str).astype(
        "category"
    )  # categories have be strings
    data["log_volume"] = np.log(data.volume + 1e-8)
    data["avg_volume_by_sku"] = data.groupby(
        ["time_idx", "sku"], observed=True
    ).volume.transform("mean")
    data["avg_volume_by_agency"] = data.groupby(
        ["time_idx", "agency"], observed=True
    ).volume.transform("mean")
    # we want to encode special days as one variable and thus need to first reverse one-hot encoding
    special_days = [
        "easter_day",
        "good_friday",
        "new_year",
        "christmas",
        "labor_day",
        "independence_day",
        "revolution_day_memorial",
        "regional_games",
        "beer_capital",
        "music_fest",
    ]
    data[special_days] = (
        data[special_days]
        .apply(lambda x: x.map({0: "-", 1: x.name}))
        .astype("category")
    )
    return data, special_days

In [ ]:
import numpy as np
data, special_days = get_stalliion_data()
time_horizon = 6  # predict six months
# make time steps first column
data["time_idx"] = data["date"].dt.year * 12 + data["date"].dt.month
data["time_idx"] -= data["time_idx"].min()
training_cutoff = data["time_idx"].max() - time_horizon
ts_col = data.pop("date")
data.insert(0, "date", ts_col.apply(lambda x:np.datetime64(x, "ns")))
# FLAML assumes input is not sorted, but we sort here for comparison purposes with y_test
data = data.sort_values(["agency", "sku", "date"])
X_train = data[lambda x: x.time_idx <= training_cutoff]
X_test = data[lambda x: x.time_idx > training_cutoff]
y_train = X_train.pop("volume")
y_test = X_test.pop("volume")

In [ ]:
X_train

In [ ]:
from flaml import AutoML
automl = AutoML()
settings = {
    "time_budget": 300,  # total running time in seconds
    "metric": "mape",  # primary metric
    "task": "ts_forecast_panel",  # task type
    "log_file_name": "stallion_forecast.log",  # flaml log file
    "eval_method": "holdout",
}
fit_kwargs_by_estimator = {
    "tft": {
        "max_encoder_length": 24,
        "static_categoricals": ["agency", "sku"],
        "static_reals": ["avg_population_2017", "avg_yearly_household_income_2017"],
        "time_varying_known_categoricals": ["special_days", "month"],
        "variable_groups": {
            "special_days": special_days
        },  # group of categorical variables can be treated as one variable
        "time_varying_known_reals": [
            "time_idx",
            "price_regular",
            "discount_in_percent",
        ],
        "time_varying_unknown_categoricals": [],
        "time_varying_unknown_reals": [
            "y",  # always need a 'y' column for the target column
            "log_volume",
            "industry_volume",
            "soda_volume",
            "avg_max_temp",
            "avg_volume_by_agency",
            "avg_volume_by_sku",
        ],
        "batch_size": 128,
        "gpu_per_trial": 0,
    }
}
"""The main flaml automl API"""
automl.fit(
    X_train=X_train,
    y_train=y_train,
    **settings,
    period=time_horizon,
    group_ids=["agency", "sku"],
    fit_kwargs_by_estimator=fit_kwargs_by_estimator,
)

In [ ]:
""" compute predictions of testing dataset """
y_pred = automl.predict(X_test)
print(y_test)
print(y_pred)

In [ ]:
""" compute different metric values on testing dataset"""
from flaml.ml import sklearn_metric_loss_score
print("mape", "=", sklearn_metric_loss_score("mape", y_pred, y_test))

def smape(y_pred, y_test):
    import numpy as np

    y_test, y_pred = np.array(y_test), np.array(y_pred)
    return round(
        np.mean(
            np.abs(y_pred - y_test) /
            ((np.abs(y_pred) + np.abs(y_test)) / 2)
        ) * 100, 2
    )

print("smape", "=", smape(y_pred, y_test))

In [ ]:
from flaml.ml import sklearn_metric_loss_score
print('flaml mape', '=', sklearn_metric_loss_score('mape', flaml_y_pred, y_test))

In [ ]:
from prophet import Prophet
prophet_model = Prophet()

In [ ]:
X_train_prophet = train_df.copy()
X_train_prophet = X_train_prophet.rename(columns={'index': 'ds', 'co2': 'y'})
prophet_model.fit(X_train_prophet)

In [ ]:
X_test_prophet = X_test.copy()
X_test_prophet = X_test_prophet.rename(columns={'index': 'ds'})
prophet_y_pred = prophet_model.predict(X_test_prophet)['yhat']
print('Predicted labels', prophet_y_pred)
print('True labels', y_test)

In [ ]:
from flaml.ml import sklearn_metric_loss_score
print('default prophet mape', '=', sklearn_metric_loss_score('mape', prophet_y_pred, y_test))

In [ ]:
from pmdarima.arima import auto_arima
import pandas as pd
import time

X_train_arima = train_df.copy()
X_train_arima.index = pd.to_datetime(X_train_arima['index'])
X_train_arima = X_train_arima.drop('index', axis=1)
X_train_arima = X_train_arima.rename(columns={'co2': 'y'})

In [ ]:
# use same search space as FLAML
start_time = time.time()
arima_model = auto_arima(X_train_arima,
                            start_p=2, d=None, start_q=1, max_p=10, max_d=10, max_q=10,
                            suppress_warnings=True, stepwise=False, seasonal=False,
                            error_action='ignore', trace=True, n_fits=650)
autoarima_y_pred = arima_model.predict(n_periods=12)
arima_time = time.time() - start_time

In [ ]:
start_time = time.time()
sarima_model = auto_arima(X_train_arima,
                            start_p=2, d=None, start_q=1, max_p=10, max_d=10, max_q=10,
                            start_P=2, D=None, start_Q=1, max_P=10, max_D=10, max_Q=10, m=12,
                            suppress_warnings=True, stepwise=False, seasonal=True,
                            error_action='ignore', trace=True, n_fits=50)
sarima_time = time.time() - start_time
autosarima_y_pred = sarima_model.predict(n_periods=12)

In [ ]:
from flaml.ml import sklearn_metric_loss_score
print('auto arima mape', '=', sklearn_metric_loss_score('mape', y_test, autoarima_y_pred))
print('auto sarima mape', '=', sklearn_metric_loss_score('mape', y_test, autosarima_y_pred))

In [ ]:
from flaml.ml import sklearn_metric_loss_score
print('flaml mape', '=', sklearn_metric_loss_score('mape', y_test, flaml_y_pred))
print('default prophet mape', '=', sklearn_metric_loss_score('mape', prophet_y_pred, y_test))
print('auto arima mape', '=', sklearn_metric_loss_score('mape', y_test, autoarima_y_pred))
print('auto sarima mape', '=', sklearn_metric_loss_score('mape', y_test, autosarima_y_pred))

In [ ]:
import matplotlib.pyplot as plt

plt.plot(X_test, y_test, label='Actual level')
plt.plot(X_test, flaml_y_pred, label='FLAML forecast')
plt.plot(X_test, prophet_y_pred, label='Prophet forecast')
plt.plot(X_test, autoarima_y_pred, label='AutoArima forecast')
plt.plot(X_test, autosarima_y_pred, label='AutoSarima forecast')
plt.xlabel('Date')
plt.ylabel('CO2 Levels')
plt.legend()
plt.show()

In [ ]:
sc.getConf().get("spark.synapse.vhd.id")